# 1.0_calculate_pt_features calculates patient features for GSPA and comparisons

In [30]:
import sys
sys.path.append('/home/aarthivenkat/Gene-Signal-Pattern-Analysis')
from gspa import calculate_wavelet_dictionary
from utils import learn_representation
from ae import run_ae
import pandas as pd
import numpy as np
import scanpy, phate
from sklearn import preprocessing

In [2]:
data = scanpy.read_h5ad('/data/lab/datasets/cpi_melanoma/processed_data.h5ad')
# subset to those samples with known response
data = data[data.obs['response'].isin(['NR', 'R'])]

In [3]:
# generate cell-cell graph
phate_op = phate.PHATE(random_state=42, verbose=False)
data_phate = phate_op.fit_transform(data.to_df())
G = phate_op.graph.to_pygsp()
del(phate_op)

## GSPA patient indicator

In [83]:
dictionary_full, wavelet_sizes = calculate_wavelet_dictionary(G, use_reduced=False)

Maximum scale: 9
Calculating Wavelets J = 9


100%|██████████| 7/7 [02:47<00:00, 23.95s/it]


In [84]:
patient_indicator = preprocessing.OneHotEncoder().fit_transform(np.array(data.obs['patient']).reshape(-1,1)).T.todense()
patient_indicator = patient_indicator / np.linalg.norm(patient_indicator, axis=1).reshape(-1,1)

In [85]:
patient_signal_reduced = learn_representation(patient_indicator, dictionary_full)
patient_signal_ae = run_ae(patient_signal_reduced)

/home/aarthivenkat/.local/lib/python3.8/site-packages/sklearn/utils/validation.py:593: FutureWarning: np.matrix usage is deprecated in 1.0 and will raise a TypeError in 1.2. Please convert to a numpy array with np.asarray. For more information see: https://numpy.org/doc/stable/reference/generated/numpy.matrix.html
  warnings.warn(


Epoch 1/100
2/2 [==============================] - 0s 84ms/step - loss: 0.0842 - val_loss: 0.0716
Epoch 2/100
2/2 [==============================] - 0s 23ms/step - loss: 0.0722 - val_loss: 0.0712
Epoch 3/100
2/2 [==============================] - 0s 22ms/step - loss: 0.0685 - val_loss: 0.0706
Epoch 4/100
2/2 [==============================] - 0s 22ms/step - loss: 0.0635 - val_loss: 0.0693
Epoch 5/100
2/2 [==============================] - 0s 20ms/step - loss: 0.0575 - val_loss: 0.0668
Epoch 6/100
2/2 [==============================] - 0s 25ms/step - loss: 0.0503 - val_loss: 0.0628
Epoch 7/100
2/2 [==============================] - 0s 21ms/step - loss: 0.0438 - val_loss: 0.0590
Epoch 8/100
2/2 [==============================] - 0s 25ms/step - loss: 0.0386 - val_loss: 0.0551
Epoch 9/100
2/2 [==============================] - 0s 24ms/step - loss: 0.0353 - val_loss: 0.0514
Epoch 10/100
2/2 [==============================] - 0s 27ms/step - loss: 0.0328 - val_loss: 0.0492
Epoch 11/100
2/2 [=

Epoch 84/100
2/2 [==============================] - 0s 21ms/step - loss: 0.0039 - val_loss: 0.0417
Epoch 85/100
2/2 [==============================] - 0s 24ms/step - loss: 0.0039 - val_loss: 0.0419
Epoch 86/100
2/2 [==============================] - 0s 25ms/step - loss: 0.0038 - val_loss: 0.0421
Epoch 87/100
2/2 [==============================] - 0s 22ms/step - loss: 0.0038 - val_loss: 0.0417
Epoch 88/100
2/2 [==============================] - 0s 21ms/step - loss: 0.0038 - val_loss: 0.0416
Epoch 89/100
2/2 [==============================] - 0s 22ms/step - loss: 0.0037 - val_loss: 0.0415
Epoch 90/100
2/2 [==============================] - 0s 26ms/step - loss: 0.0037 - val_loss: 0.0415
Epoch 91/100
2/2 [==============================] - 0s 21ms/step - loss: 0.0037 - val_loss: 0.0414
Epoch 92/100
2/2 [==============================] - 0s 21ms/step - loss: 0.0036 - val_loss: 0.0414
Epoch 93/100
2/2 [==============================] - 0s 21ms/step - loss: 0.0036 - val_loss: 0.0411
Epoch 94/1

In [101]:
np.save('results/GSPA_patient.npy', patient_signal_ae)

## GSPA+QR gene signals

In [ ]:
dictionary, wavelet_sizes = calculate_wavelet_dictionary(G)

In [53]:
all_patients_reduced = []

In [63]:
for i,pt in enumerate(np.unique(data.obs['patient'])):
    patient = data[data.obs['patient'] == pt].to_df().T.values
    pat_gene_reduced = learn_representation(patient, dictionary[data.obs['patient'] == pt])
    all_patients_reduced.append(pat_gene_reduced[:, :5])

0 Post_P1
1 Post_P10
2 Post_P11
3 Post_P12
4 Post_P13
5 Post_P14
6 Post_P15
7 Post_P16
8 Post_P17
9 Post_P18
10 Post_P19
11 Post_P1_2
12 Post_P2
13 Post_P20
14 Post_P21
15 Post_P22
16 Post_P23
17 Post_P23_2
18 Post_P28
19 Post_P28_2
20 Post_P3
21 Post_P30
22 Post_P3_2
23 Post_P4
24 Post_P5
25 Post_P5_2
26 Post_P6
27 Post_P7
28 Post_P8
29 Pre_P1
30 Pre_P12
31 Pre_P15
32 Pre_P2
33 Pre_P20
34 Pre_P24
35 Pre_P25
36 Pre_P26
37 Pre_P27
38 Pre_P28
39 Pre_P29
40 Pre_P3
41 Pre_P31
42 Pre_P33
43 Pre_P35
44 Pre_P4
45 Pre_P6
46 Pre_P7
47 Pre_P8


In [102]:
np.save('results/GSPA_gene.npy', np.array(all_patients_reduced))

## Mean expression

In [93]:
mean_patients = []
for patient in np.unique(data.obs['patient']):
    pat = data[data.obs['patient'] == patient]
    mean_patients.append(pat.to_df().mean(axis=0))

In [105]:
np.save('results/mean.npy', np.array(mean_patients))

## Cluster proportion, CD8

In [107]:
data_cd8 = data[data.obs['isCD8'], :]
patient_proportions = pd.DataFrame(index=np.unique(data_cd8.obs['patient']), columns=np.unique(data_cd8.obs['cluster_cd8']))
for patient in np.unique(data_cd8.obs['patient']):
    pat = data_cd8[data_cd8.obs['patient'] == patient]
    pat_prop = pat.obs['cluster_cd8'].value_counts() / pat.n_obs
    patient_proportions.loc[patient] = pat_prop

patient_proportions.fillna(0, inplace=True)

In [113]:
np.save('results/proportion_cd8.npy', np.array(patient_proportions))

## Cluster proportion, all

In [114]:
patient_proportions = pd.DataFrame(index=np.unique(data.obs['patient']), columns=np.unique(data.obs['cluster_all']))
for patient in np.unique(data.obs['patient']):
    pat = data[data.obs['patient'] == patient]
    pat_prop = pat.obs['cluster_all'].value_counts() / pat.n_obs
    patient_proportions.loc[patient] = pat_prop

patient_proportions.fillna(0, inplace=True)

In [115]:
np.save('results/proportion_all.npy', np.array(patient_proportions))